# Notebook 2 — The M-I decomposition in detail

Deep dive into the memory–innovation (M-I) decomposition: how Δ and κ are computed,
why they are orthogonal, and what they read for different processes.

From *Symbolic Pure Time* (Vázquez Broquá, 2026), §3.1.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..', '..', '..', 'lib'))
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import gsd

rng = np.random.default_rng(2026)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})


## The M-I decomposition: step by step

Given a time series y, SPTLS fits:

    q(t+1) = M̂ · q(t) + r_t,   q(t) = (y_t, Δy_t, Δ²y_t)

**Memory axis Δ:**
M̂^iid is SPTLS fitted on a random permutation of y (destroys temporal order, preserves
marginal distribution). Averaging over several shuffles gives the baseline that absorbs
only the mechanical autocorrelation from differencing (Δy of an i.i.d. series ≠ i.i.d.).
Δ = ‖M̂ − M̂^iid‖_F measures dependence *beyond this mechanical component*.
In population, M̂^iid → 0 as T → ∞, so Δ → ‖M̂‖_F.

**Innovation axis κ:**
r_t = q(t+1) − M̂·q(t) is the Wold innovation sequence of the kinematic embedding.
κ = excess kurtosis of {r_t[1]} (velocity-row residuals) — the distributional character
of what arrives as genuinely new at each step.

**Orthogonality:**
By the OLS projection theorem, r_t ⊥ span{q(t)}, so M̂ and {r_t} carry non-overlapping
information. Δ (a norm of M̂) and κ (a statistic of {r_t}) are orthogonal channels.


In [ ]:
def sptls_mi_verbose(y, n_shuffles=10, seed=42):
    """
    Full M-I decomposition with intermediate objects.
    Returns: delta, kappa, M, M_iid, vel_resid
    """
    res = gsd.SPTLS().fit(y)
    M = res.M.copy()

    rng_s = np.random.default_rng(seed)
    M_iid = np.zeros_like(M)
    for _ in range(n_shuffles):
        ys = y.copy(); rng_s.shuffle(ys)
        M_iid += gsd.SPTLS().fit(ys).M / n_shuffles

    delta = np.linalg.norm(M - M_iid, 'fro')
    vel_resid = res.resid_jet[:, 1]
    kappa = stats.kurtosis(vel_resid, fisher=True)
    return delta, kappa, M, M_iid, vel_resid

# Demonstrate on an I(1) AR(2) series
T = 800
y = np.zeros(T)
for t in range(2, T): y[t] = 1.6*y[t-1] - 0.6*y[t-2] + rng.standard_normal()

delta, kappa, M, M_iid, vel_resid = sptls_mi_verbose(y)

print("M̂ (3×3 operator):")
for row in M[:3,:3]: print("  ", np.round(row, 4))
print()
print("M̂^iid (shuffle baseline, 10 shuffles):")
for row in M_iid[:3,:3]: print("  ", np.round(row, 4))
print()
print(f"Δ = ‖M̂ − M̂^iid‖_F = {delta:.4f}  (memory axis)")
print(f"κ = excess kurtosis of velocity residuals = {kappa:.4f}  (innovation axis)")
print()
print("Eigenvalues of M̂:", np.round(np.linalg.eigvals(M[:3,:3]), 4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Matrix visualization: M and M_iid
vmax = max(np.abs(M[:3,:3]).max(), np.abs(M_iid[:3,:3]).max())
im0 = axes[0].imshow(M[:3,:3], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('M̂', fontweight='bold')
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(['y','Δy','Δ²y']); axes[0].set_yticklabels(['y','Δy','Δ²y'])
plt.colorbar(im0, ax=axes[0], fraction=0.046)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=9)

im1 = axes[1].imshow(M_iid[:3,:3], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('M̂^iid  (shuffle baseline)', fontweight='bold')
axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
axes[1].set_xticklabels(['y','Δy','Δ²y']); axes[1].set_yticklabels(['y','Δy','Δ²y'])
plt.colorbar(im1, ax=axes[1], fraction=0.046)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{M_iid[i,j]:.2f}', ha='center', va='center', fontsize=9)

# velocity residuals
axes[2].hist(vel_resid, bins=45, color='#4f46e5', alpha=0.75, density=True)
xg = np.linspace(vel_resid.min(), vel_resid.max(), 200)
axes[2].plot(xg, stats.norm.pdf(xg, vel_resid.mean(), vel_resid.std()), 'k--', lw=1.5, label='N(μ,σ)')
axes[2].set_title(f'Velocity-row residuals\nκ = {kappa:.3f}', fontweight='bold')
axes[2].set_xlabel('r_t[1]  (innovation at step t)'); axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_mi_detail.png', dpi=130, bbox_inches='tight')
plt.show()


## Orthogonality check

The residuals r_t are orthogonal to the design (the kinematic jet) by the OLS projection
theorem. This means Δ and κ carry non-overlapping information.


In [ ]:
res = gsd.SPTLS().fit(y)
# Check: Q0^T R ≈ 0 (projection theorem)
q0 = res._q0                  # design matrix (kinematic jet at t)
r  = res.resid_jet             # residuals

XtR = q0.T @ r                # should be ≈ 0 (3x3)
print("Q₀ᵀ · R  (should be ≈ 0 by projection theorem):")
print(np.round(XtR, 6))
print()
print(f"‖Q₀ᵀ · R‖_F = {np.linalg.norm(XtR):.2e}  (machine-precision zero)")
print()
print("→ Memory channel Δ (norm of M̂) and innovation channel κ (statistic of R)")
print("  are orthogonal: they live in disjoint subspaces of information.")


## Rolling M-I over time

A rolling-window M-I decomposition tracks how the memory and innovation structure
of a series evolve — useful for regime detection. Each window delivers a (Δ, κ) pair.


In [ ]:
def rolling_mi(y, window=120, step=20, n_shuffles=6, seed=0):
    """Rolling M-I decomposition. Returns arrays of (t, delta, kappa)."""
    ts, deltas, kappas = [], [], []
    for start in range(0, len(y) - window, step):
        yw = y[start:start+window]
        if len(yw) < 20: break
        try:
            d, k, *_ = sptls_mi_verbose(yw, n_shuffles=n_shuffles, seed=seed)
            ts.append(start + window//2)
            deltas.append(d); kappas.append(k)
        except Exception:
            pass
    return np.array(ts), np.array(deltas), np.array(kappas)

# Two-regime synthetic: stationary AR(1) transitions to I(1) random walk at t=400
T = 800
y_regime = np.zeros(T)
for t in range(1, 400):   y_regime[t] = 0.6*y_regime[t-1] + rng.standard_normal()
for t in range(400, T):   y_regime[t] = y_regime[t-1] + rng.standard_normal()

ts, deltas, kappas = rolling_mi(y_regime, window=100, step=15)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)
axes[0].plot(y_regime, color='#4f46e5', lw=0.8)
axes[0].axvline(400, color='#e0552b', lw=1.5, ls='--', label='regime change at t=400')
axes[0].set_title('Series: AR(1) → Random walk at t=400', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylabel('y_t')

axes[1].plot(ts, deltas, color='#4f46e5', lw=1.8, marker='o', ms=4)
axes[1].axvline(400, color='#e0552b', lw=1.5, ls='--')
axes[1].set_title('Memory axis Δ  (rolling, window=100)', fontweight='bold')
axes[1].set_ylabel('Δ')

axes[2].plot(ts, kappas, color='#e0552b', lw=1.8, marker='o', ms=4)
axes[2].axvline(400, color='#e0552b', lw=1.5, ls='--')
axes[2].axhline(0, color='#ccc', lw=1)
axes[2].set_title('Innovation axis κ  (rolling, window=100)', fontweight='bold')
axes[2].set_ylabel('κ'); axes[2].set_xlabel('t (centre of window)')

plt.tight_layout()
plt.savefig('fig_mi_rolling.png', dpi=130, bbox_inches='tight')
plt.show()
print("Memory axis Δ rises sharply at the regime boundary (AR(1) → I(1)).")
print("Innovation axis κ stays near zero — both regimes have Gaussian innovations.")
